# Analisis Heuristik & Eksperimen EDA (Exploratory Data Analysis)
### Topik: Optimasi Rute Pengiriman Barang Multi-Drop (Bebas Arah)

Notebook ini berisi analisis tahap awal (EDA) dari dataset pengiriman, konstruksi complete graph, perhitungan matriks jarak, serta pembuktian validitas fungsi heuristik Euclidean untuk algoritma **A* Search** dan **Greedy Best-First Search**.

## 1. Exploratory Data Analysis (EDA)
Pada tahap ini, kita akan memuat data titik pengiriman (`locations.csv`) ke dalam pandas DataFrame untuk memeriksa koordinat $X, Y$ serta beban permintaan (*demand*) paket di tiap lokasi.

In [ ]:
import sys
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Tambahkan path parent folder agar module src bisa di-import
sys.path.insert(0, os.path.abspath('..'))

# Load dataset
df = pd.read_csv('../data/raw/locations.csv')
df

In [ ]:
# Tampilkan statistik deskriptif dari sebaran lokasi dan demand
print("=== Statistik Deskriptif ===")
print(df.describe())

print("\n=== Info Tipe Data ===")
print(df.info())

print("\n=== Jumlah Tipe Lokasi ===")
print(df['type'].value_counts())

## 2. Graph Construction & Visualisasi
Kita akan menggunakan modul `DeliveryGraph` yang membungkus `networkx` untuk membangun jaringan lengkap (*complete graph*) di mana setiap node terhubung ke node lainnya.

In [ ]:
from src.graph import build_graph_from_csv
from src.visualization import plot_locations, plot_full_graph, plot_distance_heatmap

# Bangun DeliveryGraph
g = build_graph_from_csv('../data/raw/locations.csv')
print(f"Jumlah Node : {len(g.all_node_ids())}")
print(f"ID Depot    : {g.depot_id}")
print(f"Drop Points : {g.drop_point_ids()}")

In [ ]:
# Visualisasi posisi Depot dan Drop Points
fig_loc = plot_locations(g, title="Peta Posisi Depot & Drop Points")
plt.show()

In [ ]:
# Visualisasi Complete Graph (Semua Edge Terhubung)
fig_full = plot_full_graph(g, title="Visualisasi Complete Graph (Bebas Arah)")
plt.show()

In [ ]:
# Tampilkan heatmap matriks jarak Euclidean antar semua titik
fig_heat = plot_distance_heatmap(g, title="Heatmap Matriks Jarak Euclidean")
plt.show()

## 3. Pembuktian Validitas Heuristik (Admissibility & Consistency)

Agar algoritma **A* Search** dapat menghasilkan rute yang **optimal secara absolut** (jarak terpendek), fungsi heuristik $h(n)$ yang digunakan wajib memenuhi dua syarat:
1. **Admissible (Layak)**: $h(n) \le h^*(n)$ di mana $h^*(n)$ adalah biaya asli dari $n$ ke tujuan.
   - *Bukti*: Karena pergerakan diasumsikan bebas arah di ruang 2D, jarak terpendek matematis antara dua titik adalah garis lurus (jarak Euclidean). Oleh karena itu, estimasi heuristik Euclidean $h(n)$ akan selalu sama dengan jarak aktual terpendek antara dua titik ($h(n) == h^*(n)$). Estimasi tidak pernah melebihi jarak asli (*overestimate*). Q.E.D.

2. **Consistent (Konsisten)**: Untuk setiap tetangga $n'$ dari $n$, estimasi ke tujuan tidak boleh lebih besar dari biaya ke $n'$ ditambah estimasi dari $n'$ ke tujuan:
   $$h(n) \le \text{cost}(n, n') + h(n')$$
   - *Bukti*: Ini merupakan sifat matematis **Triangle Inequality** (Pertidaksamaan Segitiga): Sisi segitiga $AC \le AB + BC$. Kita akan membuktikan sifat konsistensi ini secara empiris pada seluruh kombinasi triple node dalam dataset kita menggunakan fungsi penguji bawaan.

In [ ]:
# Pengujian empiris Triangle Inequality (Consistency)
is_consistent = g.verify_triangle_inequality()
print(f"Apakah fungsi heuristik Euclidean konsisten di semua node? : {is_consistent}")
assert is_consistent == True, "Heuristik harus konsisten!"

## 4. Kesimpulan Hasil Eksperimen
1. Dataset memuat 1 Depot dan 8 Drop Points dengan koordinat bervariasi di area 100x100.
2. Matriks jarak membuktikan bahwa jarak antar titik sangat bervariasi (maksimum mendekati 100 unit).
3. Fungsi heuristik Euclidean terbukti valid secara teoretis dan teruji konsisten secara empiris, menjamin bahwa algoritma A* yang diimplementasikan akan menghasilkan jalur pengiriman paling optimal.